In [1]:
!pip install -q google-generativeai groq mistralai tavily-python

print("✅ Required packages installed")


import os
import time
from IPython.display import Markdown, display, clear_output
import google.generativeai as genai
from groq import Groq
from mistralai.client import MistralClient
from tavily import TavilyClient

print("✅ Libraries imported")


GEMINI_API_KEY = "GEMINI_API_KEY"
GROQ_API_KEY = "GROQ_API_KEY"
MISTRAL_API_KEY = "MISTRAL_API_KEY"
TAVILY_API_KEY = "TAVILY_API_KEY"

os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

print("✅ API keys configured")
print(f"   Gemini: {'✓' if GEMINI_API_KEY else '✗'}")
print(f"   Groq: {'✓' if GROQ_API_KEY else '✗'}")
print(f"   Mistral: {'✓' if MISTRAL_API_KEY else '✗'}")
print(f"   Tavily: {'✓' if TAVILY_API_KEY else '✗'}")


print("\n" + "="*50)
print("INITIALIZING AI SERVICES")
print("="*50)

available_services = []


print("\n🟡 Testing Gemini API...")
try:
    genai.configure(api_key=GEMINI_API_KEY)

    model_found = False
    for model_name in ['gemini-1.5-flash', 'gemini-pro', 'models/gemini-1.5-flash']:
        try:
            gemini_model = genai.GenerativeModel(model_name)

            test_response = gemini_model.generate_content("test")
            print(f"  ✅ Gemini {model_name} is available")
            available_services.append({'name': 'Gemini', 'client': gemini_model, 'type': 'gemini', 'model': model_name})
            model_found = True
            break
        except:
            continue

    if not model_found:
        print("  ⚠️ Could not find a working Gemini model")
except Exception as e:
    print(f"  ❌ Gemini Error: {e}")


print("\n🔵 Testing Groq API...")
try:
    groq_client = Groq(api_key=GROQ_API_KEY)

    models_to_try = ['mixtral-8x7b-32768', 'llama-3.3-70b-versatile', 'gemma2-9b-it']
    for model in models_to_try:
        try:
            test_response = groq_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": "test"}],
                max_tokens=5
            )
            print(f"  ✅ Groq {model} is available")
            available_services.append({'name': 'Groq', 'client': groq_client, 'type': 'groq', 'model': model})
            break
        except:
            continue
except Exception as e:
    print(f"  ❌ Groq Error: {e}")


print("\n🔴 Testing Mistral API...")
try:
    mistral_client = MistralClient(api_key=MISTRAL_API_KEY)
    test_response = mistral_client.chat.completions.create(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": "test"}],
        max_tokens=5
    )
    print("  ✅ Mistral Small is available")
    available_services.append({'name': 'Mistral', 'client': mistral_client, 'type': 'mistral', 'model': 'mistral-small-latest'})
except Exception as e:
    print(f"  ❌ Mistral Error: {e}")


print("\n🟣 Testing Tavily API...")
try:
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    test_response = tavily_client.search("personal finance", max_results=1)
    print("  ✅ Tavily is available")
    available_services.append({'name': 'Tavily', 'client': tavily_client, 'type': 'tavily'})
except Exception as e:
    print(f"  ❌ Tavily Error: {e}")


print("\n" + "="*50)
chat_services = [s for s in available_services if s['type'] in ['groq', 'gemini', 'mistral']]
if chat_services:

    priority = {'groq': 1, 'gemini': 2, 'mistral': 3}
    chat_services.sort(key=lambda x: priority.get(x['type'], 99))
    active_service = chat_services[0]
    print(f"✅ ACTIVE CHAT SERVICE: {active_service['name']} ({active_service['type']})")
    if 'model' in active_service:
        print(f"   Model: {active_service['model']}")
else:
    print("❌ NO CHAT SERVICES AVAILABLE")
    active_service = None


FINANCE_SYSTEM_PROMPT = """You are a certified personal finance advisor with 15 years of experience.

Your role is to provide clear, practical, and accurate financial advice on these topics:
- Budgeting and saving money
- Investing (stocks, bonds, mutual funds, real estate, crypto)
- Retirement planning (401k, IRA, pension, Social Security)
- Tax strategies and deductions
- Debt management (credit cards, loans, mortgages, student loans)
- Insurance (health, life, auto, home, disability)
- Financial goal setting
- Emergency funds
- Credit scores and reports
- College savings (529 plans, education savings)
- Real estate and mortgages
- Side hustles and additional income
- Financial independence (FIRE movement)
- Estate planning (wills, trusts)
- Banking and high-yield savings
- Stock market basics
- Dividend investing
- Index funds and ETFs

GUIDELINES:
1. Keep answers concise (2-4 sentences for basic questions, up to 6 for complex ones)
2. Use simple, everyday language - avoid jargon unless explained
3. Be conservative with investment advice - emphasize diversification and long-term thinking
4. Include relevant examples when helpful
5. Add disclaimer for complex situations: "Consider consulting a financial advisor"
6. Never promise specific returns on investments
7. Focus ONLY on personal finance - politely redirect unrelated questions

DISCLAIMER: This information is for educational purposes only. Always do your own research or consult a qualified professional before making financial decisions.
"""

print("\n✅ Personal Finance System Prompt loaded")


FINANCE_QUESTIONS = {
    "Budgeting & Saving": [
        "How do I create a monthly budget?",
        "What is the 50/30/20 budgeting rule?",
        "How much should I save each month?",
        "What are the best budgeting apps?",
        "How can I save money on groceries?",
        "What is an emergency fund and why do I need one?",
        "How much should I have in my emergency fund?",
        "How can I stop living paycheck to paycheck?",
        "What are high-yield savings accounts?",
        "How do I track my daily expenses?"
    ],

    "Debt Management": [
        "What's the best way to pay off credit card debt?",
        "Should I use the snowball or avalanche method for debt?",
        "How does debt consolidation work?",
        "What's a good credit score?",
        "How can I improve my credit score quickly?",
        "Should I pay off debt or invest?",
        "How do student loans work?",
        "What is debt-to-income ratio?",
        "How can I get out of debt fast?",
        "Should I refinance my loans?"
    ],

    "Investing Basics": [
        "How do I start investing with $1000?",
        "What's the difference between stocks and bonds?",
        "What are index funds and ETFs?",
        "How does compound interest work?",
        "What is dollar-cost averaging?",
        "Should I invest in individual stocks or mutual funds?",
        "What is a brokerage account?",
        "How do dividends work?",
        "What is the stock market?",
        "How do I choose my first investment?"
    ],

    "Retirement Planning": [
        "How much should I save for retirement?",
        "What's the difference between 401k and IRA?",
        "What is a Roth IRA?",
        "When should I start saving for retirement?",
        "What is Social Security and how does it work?",
        "Can I retire early?",
        "What is the 4% rule for retirement?",
        "Should I use a traditional or Roth 401k?",
        "What happens to my 401k if I change jobs?",
        "How do pensions work?"
    ],

    "Real Estate & Housing": [
        "Should I rent or buy a house?",
        "How much house can I afford?",
        "What is a mortgage?",
        "What's the difference between fixed and adjustable rates?",
        "How much should I save for a down payment?",
        "What are closing costs?",
        "Is real estate a good investment?",
        "Should I get a 15-year or 30-year mortgage?",
        "What is PMI and how do I avoid it?",
        "How does home equity work?"
    ],

    "Taxes": [
        "How do taxes work for freelancers?",
        "What tax deductions can I claim?",
        "What is tax-loss harvesting?",
        "Should I itemize or take standard deduction?",
        "How do capital gains taxes work?",
        "What is a tax-advantaged account?",
        "How can I reduce my tax bill?",
        "What is the difference between marginal and effective tax rate?",
        "Do I need to pay quarterly taxes?",
        "What tax credits are available?"
    ],

    "Insurance": [
        "What types of insurance do I need?",
        "How does health insurance work?",
        "What is life insurance and do I need it?",
        "Term vs whole life insurance: which is better?",
        "How much car insurance do I need?",
        "What does renters insurance cover?",
        "What is disability insurance?",
        "How do insurance deductibles work?",
        "Should I get umbrella insurance?",
        "What is long-term care insurance?"
    ],

    "Financial Planning": [
        "How do I set financial goals?",
        "What is financial independence?",
        "How do I create a net worth statement?",
        "What should be in my financial plan?",
        "How often should I review my finances?",
        "What is asset allocation?",
        "How do I diversify my investments?",
        "What is rebalancing a portfolio?",
        "How do I protect my assets?",
        "What is estate planning and do I need it?"
    ],

    "Side Hustles & Income": [
        "What are good side hustle ideas?",
        "How do I start freelancing?",
        "Should I start a business?",
        "How do I negotiate a raise?",
        "What is passive income?",
        "How can I make money online?",
        "What is the gig economy?",
        "How do I balance a side hustle with full-time work?",
        "What taxes do I pay on side hustle income?",
        "How do I turn a hobby into income?"
    ],

    "Major Life Events": [
        "How do I financially prepare for marriage?",
        "How should I manage finances with a partner?",
        "What are the costs of having a baby?",
        "How do I save for my child's education?",
        "What is a 529 plan?",
        "How do I buy a car financially wisely?",
        "What are the financial implications of divorce?",
        "How do I financially prepare for a career change?",
        "What should I do if I lose my job?",
        "How do I plan for a sabbatical?"
    ]
}

print("\n" + "="*60)
print("💰 50+ PERSONAL FINANCE QUESTIONS BY CATEGORY 💰".center(60))
print("="*60)

for category, questions in FINANCE_QUESTIONS.items():
    print(f"\n📌 {category}:")
    for i, q in enumerate(questions, 1):
        print(f"   {i}. {q}")

print("\n" + "="*60)
print("💡 Total Questions: 100+ across 10 categories")
print("="*60)


if active_service:
    print("\n" + "="*60)
    print("💰 PERSONAL FINANCE Q&A SYSTEM 💰".center(60))
    print("="*60)
    print(f"🟢 Active AI: {active_service['name']}")
    print("📋 Categories Available:")
    for i, category in enumerate(FINANCE_QUESTIONS.keys(), 1):
        print(f"   {i}. {category}")
    print("-"*60)
    print("Commands:")
    print("   'list' - Show all categories")
    print("   'cat [number]' - Show questions in category (e.g., 'cat 1')")
    print("   'q [number]' - Ask question by number (e.g., 'q 5')")
    print("   'help' - Show this menu")
    print("   'quit' - Exit")
    print("-"*60)

    conversation_history = []

    while True:
        try:

            user_input = input("\n💬 Your question/command: ").strip()


            if user_input.lower() in ['quit', 'exit', 'bye', 'thanks', 'thank you']:
                print("\n👋 Thank you for using the Personal Finance Q&A system!")
                print("Remember: Always consult a professional for specific financial advice.")
                break


            if user_input.lower() == 'help':
                print("\n📋 COMMANDS:")
                print("   'list' - Show all categories")
                print("   'cat [number]' - Show questions in category (e.g., 'cat 1')")
                print("   'q [number]' - Ask question by number (e.g., 'q 5')")
                print("   Or just type your question directly!")
                continue


            if user_input.lower() == 'list':
                print("\n📋 CATEGORIES:")
                for i, category in enumerate(FINANCE_QUESTIONS.keys(), 1):
                    print(f"   {i}. {category}")
                continue


            if user_input.lower().startswith('cat '):
                try:
                    cat_num = int(user_input.split()[1])
                    categories = list(FINANCE_QUESTIONS.keys())
                    if 1 <= cat_num <= len(categories):
                        category = categories[cat_num-1]
                        print(f"\n📌 {category} Questions:")
                        for i, q in enumerate(FINANCE_QUESTIONS[category], 1):
                            print(f"   {i}. {q}")
                    else:
                        print("❌ Invalid category number")
                except:
                    print("❌ Usage: cat [number] (e.g., 'cat 1')")
                continue


            if user_input.lower().startswith('q '):
                try:
                    q_num = int(user_input.split()[1])

                    all_questions = []
                    for cat_questions in FINANCE_QUESTIONS.values():
                        all_questions.extend(cat_questions)

                    if 1 <= q_num <= len(all_questions):
                        user_query = all_questions[q_num-1]
                        print(f"📝 Selected: {user_query}")
                    else:
                        print("❌ Invalid question number")
                        continue
                except:
                    print("❌ Usage: q [number] (e.g., 'q 5')")
                    continue
            else:
                user_query = user_input


            if not user_query:
                continue


            conversation_history.append({"role": "user", "content": user_query})
            if len(conversation_history) > 6:
                conversation_history = conversation_history[-6:]


            print("   Thinking...", end="", flush=True)
            start_time = time.time()

            if active_service['type'] == 'groq':
                messages = [{"role": "system", "content": FINANCE_SYSTEM_PROMPT}]
                for msg in conversation_history[-4:]:
                    messages.append(msg)

                response = active_service['client'].chat.completions.create(
                    model=active_service['model'],
                    messages=messages,
                    temperature=0.7,
                    max_tokens=400
                )
                answer = response.choices[0].message.content

            elif active_service['type'] == 'gemini':
                context = FINANCE_SYSTEM_PROMPT + "\n\n"
                for msg in conversation_history[-4:]:
                    if msg["role"] == "user":
                        context += f"User: {msg['content']}\n"
                    else:
                        context += f"Assistant: {msg['content']}\n"
                context += f"\nCurrent question: {user_query}"

                response = active_service['client'].generate_content(context)
                answer = response.text

            elif active_service['type'] == 'mistral':
                messages = [{"role": "system", "content": FINANCE_SYSTEM_PROMPT}]
                for msg in conversation_history[-4:]:
                    messages.append(msg)

                response = active_service['client'].chat.completions.create(
                    model=active_service['model'],
                    messages=messages,
                    temperature=0.7,
                    max_tokens=400
                )
                answer = response.choices[0].message.content

            response_time = time.time() - start_time


            print(f"\r💡 Answer ({response_time:.1f}s):\n{answer}\n")


            conversation_history.append({"role": "assistant", "content": answer})

        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"\r❌ Error: {e}")
            print("   Please try a different question.")


print("\n" + "="*60)
print("📊 QUICK FINANCE CALCULATORS".center(60))
print("="*60)
print("1. Compound Interest Calculator")
print("2. Emergency Fund Calculator")
print("3. Retirement Savings Estimator")
print("4. Debt Payoff Calculator")
print("5. Mortgage Affordability Calculator")
print("6. Investment Return Calculator")
print("7. Budget Calculator")
print("8. Net Worth Calculator")
print("-"*60)

calc_choice = input("Select calculator (1-8) or press Enter to skip: ").strip()

if calc_choice == "1":

    print("\n📈 COMPOUND INTEREST CALCULATOR")
    try:
        principal = float(input("Initial investment ($): "))
        rate = float(input("Annual interest rate (%): ")) / 100
        years = int(input("Number of years: "))
        contribution = float(input("Monthly contribution ($): "))


        future_value = principal * (1 + rate) ** years
        monthly_total = 0
        for month in range(years * 12):
            monthly_total = (monthly_total + contribution) * (1 + rate/12)

        total = future_value + monthly_total
        total_invested = principal + (contribution * years * 12)

        print(f"\n💰 Future value: ${total:,.2f}")
        print(f"   Total invested: ${total_invested:,.2f}")
        print(f"   Interest earned: ${total - total_invested:,.2f}")
        print(f"   Effective return: {((total/total_invested)-1)*100:.1f}%")
    except:
        print("❌ Invalid input")

elif calc_choice == "2":

    print("\n💵 EMERGENCY FUND CALCULATOR")
    try:
        monthly_expenses = float(input("Monthly expenses ($): "))
        job_security = input("Job security (high/medium/low): ").lower()

        if job_security == 'high':
            months = 3
        elif job_security == 'medium':
            months = 6
        else:
            months = 9

        emergency_fund = monthly_expenses * months
        print(f"\n💰 Recommended emergency fund: ${emergency_fund:,.2f}")
        print(f"   ({months} months of expenses based on {job_security} job security)")


        current = float(input("\nCurrent emergency savings ($): ") or "0")
        if current < emergency_fund:
            print(f"   Need to save: ${emergency_fund - current:,.2f} more")
            monthly = float(input("Monthly savings capacity ($): ") or "0")
            if monthly > 0:
                months_needed = (emergency_fund - current) / monthly
                print(f"   Time to goal: {months_needed:.1f} months")
    except:
        print("❌ Invalid input")

elif calc_choice == "3":

    print("\n👴 RETIREMENT SAVINGS ESTIMATOR")
    try:
        current_age = int(input("Current age: "))
        retirement_age = int(input("Retirement age: "))
        current_savings = float(input("Current retirement savings ($): "))
        monthly_contribution = float(input("Monthly contribution ($): "))
        expected_return = float(input("Expected annual return (%): ")) / 100

        years = retirement_age - current_age
        months = years * 12


        future_savings = current_savings * (1 + expected_return) ** years
        monthly_total = 0
        for month in range(months):
            monthly_total = (monthly_total + monthly_contribution) * (1 + expected_return/12)

        total = future_savings + monthly_total
        print(f"\n💰 Estimated retirement savings: ${total:,.2f}")


        withdrawal_rate = 0.04
        annual_income = total * withdrawal_rate
        print(f"   Sustainable annual income: ${annual_income:,.2f}")
        print(f"   Monthly income: ${annual_income/12:,.2f}")
    except:
        print("❌ Invalid input")

elif calc_choice == "4":

    print("\n💳 DEBT PAYOFF CALCULATOR")
    try:
        debt = float(input("Total debt ($): "))
        rate = float(input("Interest rate (%): ")) / 100
        monthly_payment = float(input("Monthly payment ($): "))

        months = 0
        balance = debt
        total_interest = 0

        while balance > 0 and months < 600:
            interest = balance * (rate / 12)
            total_interest += interest
            principal_paid = min(monthly_payment - interest, balance)
            balance -= principal_paid
            months += 1

            if balance <= 0:
                break

        if months < 600:
            years = months // 12
            remaining_months = months % 12
            print(f"\n✅ Debt free in {months} months ({years} years {remaining_months} months)")
            print(f"💰 Total interest paid: ${total_interest:,.2f}")
            print(f"📊 Total paid: ${debt + total_interest:,.2f}")


            if monthly_payment > 0:
                extra = float(input("\nExtra monthly payment amount ($) (optional): ") or "0")
                if extra > 0:
                    new_payment = monthly_payment + extra
                    new_months = 0
                    new_balance = debt
                    new_interest = 0

                    while new_balance > 0 and new_months < 600:
                        interest = new_balance * (rate / 12)
                        new_interest += interest
                        principal_paid = min(new_payment - interest, new_balance)
                        new_balance -= principal_paid
                        new_months += 1

                    savings = months - new_months
                    interest_saved = total_interest - new_interest
                    print(f"   With ${extra}/month extra: {new_months} months (save {savings} months)")
                    print(f"   Interest saved: ${interest_saved:,.2f}")
        else:
            print("⚠️ Payment too low to pay off debt")
    except:
        print("❌ Invalid input")

elif calc_choice == "5":

    print("\n🏠 MORTGAGE AFFORDABILITY CALCULATOR")
    try:
        annual_income = float(input("Annual income ($): "))
        monthly_debt = float(input("Monthly debt payments ($): "))
        down_payment = float(input("Down payment ($): "))
        interest_rate = float(input("Interest rate (%): ")) / 100
        loan_term = int(input("Loan term (years): "))


        max_monthly = (annual_income / 12) * 0.28

        available_for_mortgage = max_monthly - monthly_debt


        monthly_rate = interest_rate / 12
        payments = loan_term * 12

        if monthly_rate > 0:
            loan_amount = available_for_mortgage * ((1 - (1 + monthly_rate) ** -payments) / monthly_rate)
        else:
            loan_amount = available_for_mortgage * payments

        total_price = loan_amount + down_payment

        print(f"\n💰 Maximum home price: ${total_price:,.2f}")
        print(f"   Loan amount: ${loan_amount:,.2f}")
        print(f"   Monthly payment: ${available_for_mortgage:,.2f}")
        print(f"   Down payment: {down_payment/total_price*100:.1f}%")
    except:
        print("❌ Invalid input")

elif calc_choice == "6":

    print("\n📊 INVESTMENT RETURN CALCULATOR")
    try:
        initial = float(input("Initial investment ($): "))
        annual_return = float(input("Expected annual return (%): ")) / 100
        years = int(input("Investment period (years): "))

        future_value = initial * (1 + annual_return) ** years
        total_return = future_value - initial
        annualized_return = ((future_value/initial) ** (1/years) - 1) * 100

        print(f"\n💰 Future value: ${future_value:,.2f}")
        print(f"   Total return: ${total_return:,.2f}")
        print(f"   Return percentage: {(total_return/initial)*100:.1f}%")
        print(f"   Annualized return: {annualized_return:.2f}%")


        inflation = float(input("\nExpected inflation rate (%): ") or "3") / 100
        real_value = future_value / ((1 + inflation) ** years)
        print(f"   Inflation-adjusted value: ${real_value:,.2f}")
    except:
        print("❌ Invalid input")

elif calc_choice == "7":

    print("\n📝 50/30/20 BUDGET CALCULATOR")
    try:
        monthly_income = float(input("Monthly after-tax income ($): "))

        needs = monthly_income * 0.5
        wants = monthly_income * 0.3
        savings = monthly_income * 0.2

        print(f"\n💰 Monthly Budget Breakdown:")
        print(f"   Needs (50%): ${needs:,.2f}")
        print(f"   - Housing, utilities, groceries, transportation")
        print(f"   Wants (30%): ${wants:,.2f}")
        print(f"   - Dining out, entertainment, shopping")
        print(f"   Savings (20%): ${savings:,.2f}")
        print(f"   - Emergency fund, retirement, investments")


        print("\n📊 Track your actual spending:")
        actual_needs = float(input("Actual needs spending ($): ") or "0")
        actual_wants = float(input("Actual wants spending ($): ") or "0")
        actual_savings = float(input("Actual savings ($): ") or "0")

        if actual_needs > 0:
            print(f"\n   Needs: ${actual_needs:,.2f} ({actual_needs/monthly_income*100:.1f}%) - Target: 50%")
            print(f"   Wants: ${actual_wants:,.2f} ({actual_wants/monthly_income*100:.1f}%) - Target: 30%")
            print(f"   Savings: ${actual_savings:,.2f} ({actual_savings/monthly_income*100:.1f}%) - Target: 20%")
    except:
        print("❌ Invalid input")

elif calc_choice == "8":

    print("\n💰 NET WORTH CALCULATOR")
    print("\nASSETS:")
    try:
        cash = float(input("Cash & savings accounts ($): ") or "0")
        investments = float(input("Investments (stocks, bonds, 401k, IRA) ($): ") or "0")
        home = float(input("Home value ($): ") or "0")
        vehicles = float(input("Vehicles ($): ") or "0")
        other_assets = float(input("Other assets ($): ") or "0")

        total_assets = cash + investments + home + vehicles + other_assets

        print("\nLIABILITIES:")
        mortgage = float(input("Mortgage balance ($): ") or "0")
        loans = float(input("Student/car loans ($): ") or "0")
        credit_cards = float(input("Credit card debt ($): ") or "0")
        other_debt = float(input("Other debt ($): ") or "0")

        total_liabilities = mortgage + loans + credit_cards + other_debt
        net_worth = total_assets - total_liabilities

        print("\n" + "="*40)
        print(f"💰 NET WORTH: ${net_worth:,.2f}")
        print("="*40)
        print(f"Total Assets: ${total_assets:,.2f}")
        print(f"Total Liabilities: ${total_liabilities:,.2f}")
        print(f"Asset-to-Debt Ratio: {total_assets/total_liabilities:.2f}" if total_liabilities > 0 else "No debt!")
    except:
        print("❌ Invalid input")

print("\n" + "="*60)
print("✅ Notebook execution complete!")
print("💡 Run Cell 7 again to ask more finance questions")
print("💡 Use the calculators in Cell 8 for quick financial planning")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 20.7 MB/s eta 0:00:00
✅ Required packages installed


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Libraries imported
✅ API keys configured
   Gemini: ✓
   Groq: ✓
   Mistral: ✓
   Tavily: ✓

INITIALIZING AI SERVICES

🟡 Testing Gemini API...


  ⚠️ Could not find a working Gemini model

🔵 Testing Groq API...

🔴 Testing Mistral API...
  ❌ Mistral Error: This client is deprecated. To migrate to the new client, please refer to this guide: https://github.com/mistralai/client-python/blob/main/MIGRATION.md. If you need to use this client anyway, pin your version to 0.4.2.

🟣 Testing Tavily API...
  ❌ Tavily Error: Unauthorized: missing or invalid API key.

❌ NO CHAT SERVICES AVAILABLE

✅ Personal Finance System Prompt loaded

       💰 50+ PERSONAL FINANCE QUESTIONS BY CATEGORY 💰       

📌 Budgeting & Saving:
   1. How do I create a monthly budget?
   2. What is the 50/30/20 budgeting rule?
   3. How much should I save each month?
   4. What are the best budgeting apps?
   5. How can I save money on groceries?
   6. What is an emergency fund and why do I need one?
   7. How much should I have in my emergency fund?
   8. How can I stop living paycheck to paycheck?
   9. What are high-yield savings accounts?
   10. How do I track my 